In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models, Input
import json
import re

# ─────────────────────────────────────────────────────────────────────────────
# RAFEEQ — Layer 3: Egyptian Arabic Intent Classifier
#
# 11 intents that map 1-to-1 with the wheelchair's INTENT_LABELS in main.py.
# Each intent has ~15-20 phrases covering:
#   • Egyptian colloquial / Cairene slang
#   • Formal Modern Standard Arabic (MSA)
#   • Common mixed expressions
#
# NOTE: "rafeeq" and "sleep" are intentionally absent — the wake-word is
# handled by Layer 1 (KWS), and sleep/standby is not a navigation command.
# ─────────────────────────────────────────────────────────────────────────────

data = {
    "move_forward": [
        # Egyptian colloquial
        "قدام", "اطلع قدام", "حركني قدام", "دغري", "طوالي",
        "اتحرك قدام", "روح قدام", "امشي قدام", "مشي قدام",
        "سر قدام", "يلا قدام", "خطوة قدام", "اكمل",
        "تحرك قدام", "دلوقتي امشي", "تعالى قدام",
        # Formal MSA
        "للأمام", "للمقدمة", "على طول", "تقدم", "اكمل للأمام",
        "سر للأمام", "تحرك للأمام", "امضي للأمام", "انطلق",
        "تقدم للأمام",
    ],
    "move_backward": [
        # Egyptian colloquial
        "لورا", "ضهرك", "ارجع لورا", "ارجع شوية", "بضهرك",
        "هات لورا", "تعالى لورا", "تحرك لورا", "شوية لورا",
        "رجعني", "نزل شوية",
        # Formal MSA
        "ارجع ورا", "للخلف", "تراجع", "اتراجع", "رجوع",
        "تحرك للخلف", "اتحرك للخلف", "امشي للخلف",
        "الرجوع للخلف",
    ],
    "turn_right": [
        # Egyptian colloquial
        "يمين", "لف يمين", "خش يمين", "اكسر يمين", "دور يمين",
        "ادخل يمين", "روح يمين", "على اليمين", "شوية يمين",
        # Formal MSA
        "اتجه لليمين", "يمينا", "تحول يمين", "انعطف يمين",
        "تحول لليمين", "انعطف لليمين",
    ],
    "turn_left": [
        # Egyptian colloquial
        "شمال", "لف شمال", "خش شمال", "اكسر شمال", "دور شمال",
        "ادخل شمال", "روح شمال", "على الشمال", "شوية شمال",
        "يسار", "لف يسار", "دور يسار",
        # Formal MSA
        "اتجه لليسار", "يسارا", "تحول يسار", "انعطف يسار",
        "تحول لليسار", "انعطف لليسار",
    ],
    "stop": [
        # Egyptian colloquial
        "وقف", "بس", "خلاص", "كفاية", "ستوب", "اسكن",
        "وقف هنا", "توقف هنا", "ابقى هنا", "خلاص بقى",
        "هنا وقف",
        # Formal MSA
        "اثبت", "قف", "توقف", "لا تتحرك", "ابق مكانك",
        "اوقف", "توقف فورا", "ثبت",
    ],
    "go_to_bathroom": [
        # Egyptian colloquial
        "الحمام", "تواليت", "الدورة", "حمام", "باثروم",
        "وديني الحمام", "عايز اروح الحمام", "ادخل الحمام",
        "روح الحمام", "روحني الحمام", "خدني الحمام",
        "دورة المياه",
        # Formal MSA
        "اذهب للحمام", "المرحاض", "اريد الحمام",
        "توجه للحمام", "الذهاب للحمام",
    ],
    "go_to_bedroom": [
        # Egyptian colloquial
        "الاوضة", "اوضتي", "الغرفة", "عايز انام", "روح الغرفة",
        "وديني الاوضة", "روح اوضتي", "وديني الغرفة",
        "خدني الغرفة", "اوضة النوم",
        # Formal MSA
        "غرفة النوم", "اذهب للغرفة", "اريد غرفة النوم",
        "توجه لغرفة النوم",
    ],
    "go_to_living_room": [
        # Egyptian colloquial
        "الصالة", "الليفنج", "وديني الصالة", "الاوضة الكبيرة",
        "روح الصالة", "خدني الصالة", "عايز اقعد في الصالة",
        "ليفنج",
        # Formal MSA
        "قاعة المعيشة", "غرفة المعيشة", "الصالون",
        "اذهب للصالة", "توجه للصالة", "اريد الصالة",
    ],
    "go_to_kitchen": [
        # Egyptian colloquial
        "المطبخ", "الكيتشن", "وديني المطبخ", "جعان",
        "عايز اكل", "حركني للمطبخ", "روح المطبخ",
        "خدني المطبخ", "كيتشن",
        # Formal MSA
        "اذهب للمطبخ", "اريد المطبخ", "توجه للمطبخ",
        "الذهاب للمطبخ",
    ],
    "speed_up": [
        # Egyptian colloquial
        "اسرع", "زود السرعة", "بسرعة", "اسرع شوية",
        "روح بسرعة", "زيد", "تحرك اسرع", "اسرع اكثر",
        "زود", "اسرع بقى",
        # Formal MSA
        "زيادة السرعة", "اتحرك بسرعة", "رفع السرعة",
        "زيد السرعة", "اسرع قليلا",
    ],
    "slow_down": [
        # Egyptian colloquial
        "بطي", "شوية بطي", "هدى", "بطي شوية", "اهدى",
        "ابطى", "على مهلك", "بطي اكثر",
        # Formal MSA
        "تباطا", "خفف السرعة", "قلل السرعة", "ابطا السرعة",
        "تحرك ببطء", "بطء اكثر", "اتحرك ببطء",
    ],
}

print(f"Dataset loaded: {len(data)} intents")
for intent, phrases in sorted(data.items()):
    print(f"  {intent:25s}: {len(phrases):2d} phrases")

In [ ]:
def normalize_arabic(text: str) -> str:
    """
    Normalise Egyptian Arabic text before Bag-of-Words vectorisation.

    Applies 6 steps so that formal and colloquial variants of the same word
    collapse to the same root token — matching what the model was trained on.

    Steps
    -----
    1. Remove diacritics (harakat/tashkeel ً ٌ ٍ َ ُ ِ ّ ْ) and tatweel (ـ)
       so "يمينًا" → "يمينا" and "الحمّام" → "الحمام".
    2. Unify Alef variants (أ إ آ ٱ → ا)
       so "للأمام" → "للامام".
    3. Unify Ya tail (ى → ي) — very common in Egyptian Arabic word endings
       so "ابقى" → "ابقي".
    4. Unify Hamza seats (ؤ → و,  ئ → ي).
    5. Strip non-Arabic/non-Latin characters (punctuation, digits, symbols).
    6. Strip common Arabic prefixes word-by-word — longest match first so
       'بال' beats 'ال'. Guard: ≥2 chars must remain after stripping so a
       3-letter word like 'الي' is not reduced to a single letter 'ي'.

    Examples
    --------
    >>> normalize_arabic("للأمام")        # → 'امام'
    >>> normalize_arabic("الحمّام")        # → 'حمام'
    >>> normalize_arabic("يمينًا")         # → 'يمينا'
    >>> normalize_arabic("دغري")           # → 'دغري'   (unchanged)
    >>> normalize_arabic("اتجه لليسار")    # → 'اتجه يسار'
    """
    # 1. Diacritics (U+064B–U+0652), superscript Alef (U+0670), tatweel (U+0640)
    text = re.sub(r'[\u064B-\u0652\u0670\u0640]', '', text)

    # 2. Alef normalisation
    text = re.sub(r'[أإآٱ]', 'ا', text)

    # 3. Ya tail normalisation
    text = text.replace('ى', 'ي')

    # 4. Hamza seats
    text = text.replace('ؤ', 'و').replace('ئ', 'ي')

    # 5. Remove anything that isn't Arabic script, Latin letters, or whitespace
    text = re.sub(r'[^\u0600-\u06FFa-zA-Z\s]', '', text)

    # 6. Prefix stripping — ordered longest → shortest so 'بال' is tried before 'ال'
    prefixes = ['بال', 'لل', 'ال', 'في', 'و']
    cleaned = []
    for word in text.split():
        for prefix in prefixes:
            if word.startswith(prefix) and len(word) >= len(prefix) + 2:
                word = word[len(prefix):]
                break  # strip at most one prefix per word
        cleaned.append(word)

    return ' '.join(cleaned).strip()


# ── Sanity checks ────────────────────────────────────────────────────────────
tests = [
    ("للأمام",         "امام"),
    ("الحمام",         "حمام"),
    ("يمينًا",          "يمينا"),
    ("دغري",           "دغري"),
    ("المطبخ",         "مطبخ"),
    ("اتجه لليسار",    "اتجه يسار"),
    ("بالمطبخ",        "مطبخ"),
    ("الحمّام",         "حمام"),
]

print("Normalisation sanity checks:")
all_ok = True
for raw, expected in tests:
    result = normalize_arabic(raw)
    ok = result == expected
    if not ok:
        all_ok = False
    status = "✅" if ok else "❌"
    print(f"  {status}  {raw!r:20s} → {result!r:15s}  (expected: {expected!r})")

print("\n✅ All checks passed!" if all_ok else "\n❌ Some checks failed — review normalize_arabic().")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Step 1 — Build vocabulary + encode training data
# ─────────────────────────────────────────────────────────────────────────────

# Labels are sorted so their alphabetical position = model class index.
# This order is saved in metadata.json and must be reproduced exactly at
# inference time — main.py reads labels from metadata.json, never hard-codes.
labels = sorted(data.keys())

# Collect all normalised tokens to build the vocabulary
all_tokens = []
for intent in labels:
    for phrase in data[intent]:
        all_tokens.extend(normalize_arabic(phrase).split())

vocab = sorted(set(all_tokens))          # sorted → reproducible index mapping
vocab_size = len(vocab)
word2idx = {w: i for i, w in enumerate(vocab)}

print(f"Labels ({len(labels)}): {labels}")
print(f"Vocabulary size: {vocab_size} unique tokens\n")


def text_to_bow(text: str) -> np.ndarray:
    """Normalise `text` then encode as a float32 BoW vector of length vocab_size."""
    vec = np.zeros(vocab_size, dtype=np.float32)
    for token in normalize_arabic(text).split():
        if token in word2idx:
            vec[word2idx[token]] = 1.0
    return vec


# Build X (BoW vectors) and y (class indices)
X, y = [], []
for label_idx, intent in enumerate(labels):
    for phrase in data[intent]:
        X.append(text_to_bow(phrase))
        y.append(label_idx)

X = np.array(X, dtype=np.float32)
y = np.array(y, dtype=np.int32)
print(f"Training set: {X.shape[0]} samples × {X.shape[1]} features → {len(labels)} classes")


# ─────────────────────────────────────────────────────────────────────────────
# Step 2 — Define and train the model
# ─────────────────────────────────────────────────────────────────────────────

model = models.Sequential([
    Input(shape=(vocab_size,)),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(len(labels), activation='softmax'),
], name="rafeeq_intent_classifier")

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)
model.summary()

print("\n🧠 Training the Intent Classifier …")
history = model.fit(
    X, y,
    epochs=200,
    batch_size=16,
    validation_split=0.1,
    verbose=0,
)
final_acc = history.history['accuracy'][-1]
final_val = history.history.get('val_accuracy', [0])[-1]
print(f"✅  Training accuracy:   {final_acc:.1%}")
print(f"    Validation accuracy: {final_val:.1%}")


# ─────────────────────────────────────────────────────────────────────────────
# Step 3 — Export TFLite with dynamic-range quantization
#          (smaller model file, faster inference on Raspberry Pi 5)
# ─────────────────────────────────────────────────────────────────────────────

converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]   # dynamic-range quantization
tflite_model = converter.convert()

with open('text_classifier.tflite', 'wb') as f:
    f.write(tflite_model)
print(f"\nSaved: text_classifier.tflite  ({len(tflite_model) / 1024:.1f} KB)")


# ─────────────────────────────────────────────────────────────────────────────
# Step 4 — Save metadata.json (single source of truth for vocab + labels)
#
# Format:
#   "vocab":  list of tokens — list index = BoW vector index
#   "labels": list of intents — list index = model output class index
#
# main.py loads THIS file at runtime; never regenerate vocab independently.
# ─────────────────────────────────────────────────────────────────────────────

metadata = {"vocab": vocab, "labels": labels}
with open('metadata.json', 'w', encoding='utf-8') as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)
print(f"Saved: metadata.json  (vocab={vocab_size} tokens, labels={len(labels)})")


# ─────────────────────────────────────────────────────────────────────────────
# Step 5 — Smoke tests: run exported TFLite model on canonical phrases
# ─────────────────────────────────────────────────────────────────────────────

interp = tf.lite.Interpreter(model_path='text_classifier.tflite')
interp.allocate_tensors()
in_idx  = interp.get_input_details()[0]['index']
out_idx = interp.get_output_details()[0]['index']

smoke_tests = [
    # (phrase,                 expected_intent)
    ("قدام",                   "move_forward"),
    ("دغري",                   "move_forward"),
    ("للأمام",                 "move_forward"),
    ("الحمام",                 "go_to_bathroom"),
    ("وديني الحمام",           "go_to_bathroom"),
    ("تواليت",                 "go_to_bathroom"),
    ("يمين",                   "turn_right"),
    ("شمال",                   "turn_left"),
    ("وقف",                    "stop"),
    ("بطي",                    "slow_down"),
    ("اسرع",                   "speed_up"),
    ("لورا",                   "move_backward"),
    ("الصالة",                 "go_to_living_room"),
    ("المطبخ",                 "go_to_kitchen"),
    ("غرفة النوم",             "go_to_bedroom"),
]

print("\n─── Smoke Tests ────────────────────────────────────────────────────")
all_pass = True
for phrase, expected in smoke_tests:
    bow = text_to_bow(phrase).reshape(1, -1)
    interp.set_tensor(in_idx, bow)
    interp.invoke()
    probs      = interp.get_tensor(out_idx)[0]
    pred_idx   = int(np.argmax(probs))
    confidence = float(probs[pred_idx])
    predicted  = labels[pred_idx]
    ok = predicted == expected
    if not ok:
        all_pass = False
    status = "✅" if ok else "❌"
    conf_warn = "  ⚠️  low confidence" if confidence < 0.80 else ""
    print(f"  {status}  {phrase!r:25s} → {predicted:25s} ({confidence:.0%}){conf_warn}")

print()
if all_pass:
    print("✅ All smoke tests passed! model is ready for deployment.")
else:
    print("❌ Some tests failed — expand the dataset and retrain.")